# The Ultimate Merger & Cleaner

this deals with EVERYTHING (convert all images to jpg - make all txt files - merge images into one `unified folder` - use the ultimate mapper)

In [58]:
# import libraries:

import os
import json
import shutil
from PIL import Image
from natsort import natsorted # Essential for natural ordering (1, 2, 10... instead of 1, 10, 2)
# from ../src/utils/master_mapper import master_mapper (when Azzam does it)!

## Convert all images to one type (jpg)

In [59]:
# Get the old paths:
damage_img_dir = "../data/Car damages dataset/File1/img"
parts_img_dir =  "../data/Car parts dataset/File1/img"


# the output path:
output_img_dir = "../data/unified_dataset/images"

#put all the folders in a list:
source_dir = [damage_img_dir, parts_img_dir] # add healthy images path later

#make the directory if it not there:
os.makedirs(output_img_dir, exist_ok=True)

In [4]:
#start converting to jpg:


# Dictionary to link old names to new names (IMPORTANT for JSON stage)
name_mapping = {}

def convert_to_png(source_dir,output_dir,prefix="Vehicle"):
    #counter
    processed_count = 0 

    #if the file already exist , remove it first:
    if os.path.exists(output_img_dir):
        shutil.rmtree(output_img_dir)
    os.makedirs(output_img_dir, exist_ok=True)

    # Tuple of valid extentions:
    valid_extensions = ('.jpg', '.webp', '.jpeg','.png')

    #create a set to store the paths (without duplicates):
    unique_files = set()

    #A dictionary to remeber which file belongs to which folder (did the image come from parts or damage imgs)
    file_location = {}
    


    # check each folder and link its image with it
    for folder in source_dir:
        if not os.path.exists(folder) :
            continue #if the folder does not exist move on to the next folder

        for filename in os.listdir(folder):
            if filename.lower().endswith(valid_extensions):
                if filename not in unique_files:
                    unique_files.add(filename)
                    file_location[filename] = folder

    

    for filename in natsorted(unique_files):
            processed_count += 1

            # Get full original path
            full_path = os.path.join(file_location[filename],filename)


            #opne the image
            img = Image.open(full_path).convert('RGB')
            
            #make a new name for it:
            new_filename = f"{prefix}_{processed_count:04d}.png"
            target_path = os.path.join(output_dir, new_filename)

            img.save(target_path, "png")

            print(f"converted {filename} to -> {new_filename}")

            # C- Store the mapping (Old Name -> New Name)
            # We store only the stem (name without extension)
            old_stem = os.path.splitext(filename)[0]     # Car damages 2 (without the extention)
            new_stem = os.path.splitext(new_filename)[0]  # what Car damages 2 maps to -> eg. Vehicle_0001
            name_mapping[old_stem] = new_stem              # {"Car damages 2": "Vehicle_0001"}

            pass # end of .endswith if statement


    return processed_count,name_mapping #end of the function


total_images, mapping_dic =  convert_to_png(source_dir=source_dir,output_dir=output_img_dir)
print(f"Processed {total_images}")
print(f"the naming dic is: {mapping_dic}")

converted Car damages 2.jpg to -> Vehicle_0001.png
converted Car damages 11.png to -> Vehicle_0002.png
converted Car damages 12.png to -> Vehicle_0003.png
converted Car damages 13.png to -> Vehicle_0004.png
converted Car damages 14.png to -> Vehicle_0005.png
converted Car damages 15.png to -> Vehicle_0006.png
converted Car damages 16.png to -> Vehicle_0007.png
converted Car damages 17.png to -> Vehicle_0008.png
converted Car damages 20.png to -> Vehicle_0009.png
converted Car damages 21.png to -> Vehicle_0010.png
converted Car damages 22.png to -> Vehicle_0011.png
converted Car damages 23.png to -> Vehicle_0012.png
converted Car damages 24.png to -> Vehicle_0013.png
converted Car damages 25.png to -> Vehicle_0014.png
converted Car damages 26.png to -> Vehicle_0015.png
converted Car damages 27.png to -> Vehicle_0016.png
converted Car damages 28.png to -> Vehicle_0017.png
converted Car damages 29.png to -> Vehicle_0018.png
converted Car damages 30.jpg to -> Vehicle_0019.png
converted Car

## JSON Mapper

In [60]:
# Create a small slice of the dictionary for testing

small_test_map = (list(mapping_dic.items())[:5])

print(small_test_map)


[('Car damages 2', 'Vehicle_0001'), ('Car damages 11', 'Vehicle_0002'), ('Car damages 12', 'Vehicle_0003'), ('Car damages 13', 'Vehicle_0004'), ('Car damages 14', 'Vehicle_0005')]


In [61]:
master_mapper = {
    # Damages
    "Missing part": 0, "Broken part": 1, "Scratch": 2, "Cracked": 3,
    "Dent": 4, "Flaking": 5, "Paint chip": 6, "Corrosion": 7,
    # Parts (Re-indexed from 8)
    "Quarter-panel": 8, "Front-wheel": 9, "Back-window": 10, "Trunk": 11,
    "Front-door": 12, "Rocker-panel": 13, "Grille": 14, "Windshield": 15,
    "Front-window": 16, "Back-door": 17, "Headlight": 18, "Back-wheel": 19,
    "Back-windshield": 20, "Hood": 21, "Fender": 22, "Tail-light": 23,
    "License-plate": 24, "Front-bumper": 25, "Back-bumper": 26, "Mirror": 27, "Roof": 28
}

In [63]:
damage_ann_dir = "../data/Car damages dataset/File1/ann"
parts_ann_dir = "../data/Car parts dataset/File1/ann"
output_labels_dir = "../data/unified_dataset/labels"

ann_source_dir = [damage_ann_dir, parts_ann_dir]


for filename in natsorted(os.listdir(damage_ann_dir))[:2]:
    print(f"og filename is: {filename}")

    name_wihtout_json = str(os.path.splitext(filename)[0])
    print(f"name without the json is: {name_wihtout_json}")

    name_without_extention = os.path.splitext(name_wihtout_json)[0]
    print(f"final clean name is: {name_without_extention}")


    if name_without_extention in mapping_dic:
        print('found the file in the mapping_dictionary:')
        new_name = name_mapping[name_without_extention]

        print(f'the old name "{name_without_extention}" maps to -> "{new_name}"')
    print("---------------------------------------------------------")



og filename is: Car damages 2.jpg.json
name without the json is: Car damages 2.jpg
final clean name is: Car damages 2
found the file in the mapping_dictionary:
the old name "Car damages 2" maps to -> "Vehicle_0001"
---------------------------------------------------------
og filename is: Car damages 11.png.json
name without the json is: Car damages 11.png
final clean name is: Car damages 11
found the file in the mapping_dictionary:
the old name "Car damages 11" maps to -> "Vehicle_0002"
---------------------------------------------------------


In [ ]:

def normalize_annotation(source_dir, mapping_dic, mapper , output_dir):

    #Counter:
    processed = 0

    # Clean output labels dir once at the start
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    for folder in source_dir:
        if not os.path.exists(folder):
            continue

        for filename in natsorted(os.listdir(folder)):
            if filename.endswith('.json'):

                #stript the name from the extenstions 
                clean_name = os.path.splitext(os.path.splitext(filename)[0])[0]

                #if the name exist -> map it to the new name (Car damages 2 maps to -> Vehicle_0001)
                if clean_name in mapping_dic:
                    new_name = mapping_dic[clean_name]

                    #open the json file:
                    json_path = os.path.join(folder,filename)
                    with open(json_path, 'r') as f:
                        data = json.load(f)

                        img_height, img_wdith = data['size']['height'] , data['size']['width']

                        #start Normalize 

                        #list to hold the coordinates + class -> [3 0.534523452 0.35123412 ...etc]
                        yolo_lines = []
                        print(f"now processing file number: {processed}")

                        for obj in data['objects']:
                            class_name = obj['classTitle'] # important for later to be used with the master mapper

                            #list to hold nomralized yolo cords (to later be added to yolo_lines)
                            yolo_coords = []


                            #get the normalized values 
                            for coordinates in obj['points']['exterior']:
                                x_norm = round(coordinates[0] / img_wdith, 8)
                                y_norm = round(coordinates[1]/ img_height, 8)

                                #put the coords in the yolo_coords
                                yolo_coords.extend([str(x_norm) , str(y_norm)])
                                
                            #apend the class + coords (speperated by space " ")
                            mapped_class = mapper[class_name]

                            yolo_line = f"{mapped_class} " + " ".join(yolo_coords) + "\n"

                            yolo_lines.append(yolo_line)

                        #now we have everything, just need to make txt files:
                        txt_name = f"{new_name}.txt"
                        output_path = os.path.join(output_dir , txt_name)

                        #start writing to the txt files: (note that we use append not write to be able to add overlapping information to the txt file in case there was any)
                        with open(output_path, 'a') as out_f:
                            # Join all lines found in this JSON and write them
                            out_f.writelines(yolo_lines)

                    # Increment the counter after a successful file process
                    processed += 1

        
    return processed



In [70]:
# Execute Stage 2

# Paths for JSON sources
damage_ann_dir = "../data/Car damages dataset/File1/ann"
parts_ann_dir = "../data/Car parts dataset/File1/ann"
output_labels_dir = "../data/unified_dataset/labels"

ann_source_dir = [damage_ann_dir, parts_ann_dir]

#call the function
total_labels = normalize_annotation(
    source_dir=ann_source_dir, 
    mapping_dic=mapping_dic, 
    mapper=master_mapper, 
    output_dir=output_labels_dir
)

print(f"✅ DONE: {total_labels} JSON files processed and unified!")

{'height': 442, 'width': 562}
{'height': 447, 'width': 628}
{'height': 354, 'width': 629}
{'height': 354, 'width': 629}
{'height': 359, 'width': 629}
{'height': 366, 'width': 629}
{'height': 398, 'width': 629}
{'height': 402, 'width': 629}
{'height': 431, 'width': 629}
{'height': 447, 'width': 629}
{'height': 448, 'width': 629}
{'height': 452, 'width': 629}
{'height': 471, 'width': 629}
{'height': 367, 'width': 630}
{'height': 376, 'width': 630}
{'height': 437, 'width': 630}
{'height': 438, 'width': 630}
{'height': 444, 'width': 630}
{'height': 477, 'width': 624}
{'height': 576, 'width': 877}
{'height': 446, 'width': 630}
{'height': 446, 'width': 630}
{'height': 419, 'width': 637}
{'height': 440, 'width': 637}
{'height': 471, 'width': 629}
{'height': 476, 'width': 881}
{'height': 442, 'width': 637}
{'height': 444, 'width': 637}
{'height': 360, 'width': 638}
{'height': 363, 'width': 638}
{'height': 441, 'width': 638}
{'height': 358, 'width': 639}
{'height': 387, 'width': 639}
{'height':

# Garbage


In [ ]:
# Cell 3: JSON Annotation Processing & Merging
import os
import json

# Paths for JSON sources
damage_ann_dir = "../data/Car damages dataset/File1/ann"
parts_ann_dir = "../data/Car parts dataset/File1/ann"
output_labels_dir = "../data/unified_dataset/labels"

#if the file already exist , remove it first:
if os.path.exists(output_labels_dir):
    shutil.rmtree(output_labels_dir)

# Ensure output labels directory exists
os.makedirs(output_labels_dir, exist_ok=True)

# THE UNIFIED MASTER MAPPER (Assuming it's ready)
# Damages: 0-7, Parts: 8-28
master_mapper = {
    # Damages
    "Missing part": 0, "Broken part": 1, "Scratch": 2, "Cracked": 3,
    "Dent": 4, "Flaking": 5, "Paint chip": 6, "Corrosion": 7,
    # Parts (Re-indexed from 8)
    "Quarter-panel": 8, "Front-wheel": 9, "Back-window": 10, "Trunk": 11,
    "Front-door": 12, "Rocker-panel": 13, "Grille": 14, "Windshield": 15,
    "Front-window": 16, "Back-door": 17, "Headlight": 18, "Back-wheel": 19,
    "Back-windshield": 20, "Hood": 21, "Fender": 22, "Tail-light": 23,
    "License-plate": 24, "Front-bumper": 25, "Back-bumper": 26, "Mirror": 27, "Roof": 28
}

def process_annotations_v2(ann_dirs, mapping, mapper, output_dir):
    processed_labels = 0
    # List of possible extensions that might be embedded in JSON filenames
    embedded_exts = ['.png', '.jpg', '.jpeg', '.webp']
    
    # Clean mapping keys just in case
    clean_mapping = {str(k).strip(): v for k, v in mapping.items()}

    for ann_dir in ann_dirs:
        if not os.path.exists(ann_dir): continue
        print(f"📂 Processing: {ann_dir}")
        
        for json_file in os.listdir(ann_dir):
            if json_file.endswith('.json'):
                # 1. Start with filename without .json
                old_stem = os.path.splitext(json_file)[0].strip()
                
                # 2. STRIP embedded image extensions from the JSON name
                # Example: 'Car damages 101.png' -> 'Car damages 101'
                for ext in embedded_exts:
                    if old_stem.lower().endswith(ext):
                        old_stem = old_stem[: -len(ext)].strip()
                        break
                
                # 3. Match against our unique image mapping
                if old_stem in clean_mapping:
                    new_name = clean_mapping[old_stem]
                    json_path = os.path.join(ann_dir, json_file)
                    txt_path = os.path.join(output_dir, f"{new_name}.txt")
                    
                    with open(json_path, 'r') as f:
                        data = json.load(f)
                        img_w, img_h = data["size"]["width"], data["size"]["height"]
                        
                        # Open in 'a' mode to MERGE labels from multiple JSONs for the same image
                        with open(txt_path, "a") as out_f:
                            for obj in data['objects']:
                                class_name = obj["classTitle"]
                                if class_name in mapper:
                                    yolo_class_id = mapper[class_name]
                                    points = [str(yolo_class_id)]
                                    for coord in obj["points"]["exterior"]:
                                        # Standard YOLO Normalization
                                        points.extend([str(round(coord[0] / img_w, 8)), 
                                                       str(round(coord[1] / img_h, 8))])
                                    out_f.write(" ".join(points) + "\n")
                    
                    processed_labels += 1
                    
    return processed_labels

# Re-run the process
total_labels = process_annotations_v2([damage_ann_dir, parts_ann_dir], mapping_dic, master_mapper, output_labels_dir)
print(f"🚀 SUCCESS: Processed {total_labels} annotations into {output_labels_dir}")

📂 Processing: ../data/Car damages dataset/File1/ann
📂 Processing: ../data/Car parts dataset/File1/ann
🚀 SUCCESS: Processed 1812 annotations into ../data/unified_dataset/labels


In [ ]:
import os
import shutil
from PIL import Image

# 1- Define your source folders in a list
source_folders = [damage_img_dir, parts_img_dir]
# 2- Dictionary to link old names to new names (Crucial for JSON stage)
name_mapping = {}

def convert_to_png_unified(source_dirs, output_dir, prefix="Vehicle"):
    processed_count = 0 
    
    # Cleaning and preparing the output directory
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    valid_extensions = ('.jpg', '.webp', '.jpeg', '.png')
    
    # A- Use a set to collect unique filenames across all directories
    unique_files = set()
    # Dictionary to remember which folder each file belongs to
    file_location = {}

    for folder in source_dirs:
        if not os.path.exists(folder): continue
        for filename in os.listdir(folder):
            if filename.lower().endswith(valid_extensions):
                if filename not in unique_files:
                    unique_files.add(filename)
                    file_location[filename] = folder

    # B- Start processing the unique set
    for filename in sorted(list(unique_files)):
        processed_count += 1
        
        # Get full original path
        full_path = os.path.join(file_location[filename], filename)
        
        # Open and convert to RGB
        img = Image.open(full_path).convert('RGB')
        
        # Create the new standardized name
        new_filename = f"{prefix}_{processed_count:04d}.png"
        target_path = os.path.join(output_dir, new_filename)

        # Save the image
        img.save(target_path, "png")

        # C- Store the mapping (Old Name -> New Name)
        # We store only the stem (name without extension)
        old_stem = os.path.splitext(filename)[0]
        new_stem = os.path.splitext(new_filename)[0]
        name_mapping[old_stem] = new_stem

        print(f"Mapped and converted: {filename} -> {new_filename}")

    return processed_count

# Run the unified process
total = convert_to_png_unified(source_dirs=source_folders, output_dir=output_img_dir)